In [1]:
import numpy as np
from scipy.stats import norm

In [14]:
n = 100
teta = 10

sample = np.random.uniform(teta, teta * 2, n)
variation_series = sorted(sample)

beta = 0.95
alpha = 0.05

t1_precise = 1 + np.pow(alpha * 0.5, 1 / n)
t2_precise = 1 + np.pow(1 - alpha * 0.5, 1 / n)

left_bound_precise = variation_series[n - 1] / t2_precise
right_bound_precise = variation_series[n - 1] / t1_precise

print(f"teta = {teta}")
print(f"точный доверительный интервал: {left_bound_precise} < teta < {right_bound_precise}")
print(f"его длина: {right_bound_precise - left_bound_precise}")

teta = 10
точный доверительный интервал: 9.958279912378174 < teta < 10.140649862496822
его длина: 0.1823699501186482


In [3]:
def get_kth_moment(sample: np.ndarray, k: int) -> float:
    result = 0
    for i in range(sample.size):
        result += sample[i] ** k
    result /= sample.size
    return result

In [19]:
t1_asymptotic = norm.ppf(0.025)
t2_asymptotic = norm.ppf(0.975)

alpha_1 = get_kth_moment(sample, 1)
alpha_2 = get_kth_moment(sample, 2)
teta_1 = 2 / 3 * np.average(sample)
sigma_teta_1 = 2 / 3 * np.sqrt(alpha_2 - alpha_1 ** 2)

left_bound_asymptotic = teta_1 - t2_asymptotic * sigma_teta_1 / np.sqrt(n)
right_bound_asymptotic = teta_1 - t1_asymptotic * sigma_teta_1 / np.sqrt(n)

print(f"асимптотический доверительный интервал по ОММ: {left_bound_asymptotic} < teta < {right_bound_asymptotic}")
print(f"его длина: {right_bound_asymptotic - left_bound_asymptotic}")

асимптотический доверительный интервал по ОММ: 9.554683184031745 < teta < 10.274750437105354
его длина: 0.7200672530736085


In [5]:
def do_bootstrap(cnt: int, sample: np.ndarray, param_func) -> np.ndarray:
    """
    Здесь просто генерация выборок
    """
    answ = []
    for i in range(cnt):
        podviborka = []
        for j in range(sample.size):
            index = np.random.randint(0, n)
            podviborka.append(sample[index])
        podviborka = np.array(podviborka)
        answ.append(param_func(podviborka))
    return np.array(answ)

In [6]:
def get_teta(sample):
    return 2 / 3 * np.average(sample)

In [ ]:
model_teta = teta_1

bootstrap_tetas = do_bootstrap(1000, sample, get_teta)
deltas = bootstrap_tetas - model_teta

variation_series_delta = sorted(deltas)

k1 = int((1 - beta) * 0.5 * 1000)
k2 = int((1 + beta) * 0.5 * 1000)

left_bound_unparam_boots = teta_1 - variation_series_delta[k2]
right_bound_unparam_boots = teta_1 - variation_series_delta[k1]
print(f"бустраповский (непараметрический) доверительный интервал по ОММ: {left_bound_unparam_boots} < teta < {right_bound_unparam_boots}")
print(f"его длина: {right_bound_unparam_boots - left_bound_unparam_boots}")

бустраповский (непараметрический) доверительный интервал по ОММ: 9.540960597865482 < teta < 10.274810082819712
его длина: 0.7338494849542307


In [8]:
def do_parametric_bootstrap(cnt: int, sample: np.ndarray, param_func, teta: float) -> np.ndarray:
    """
    Здесь просто генерация выборок
    """
    answ = []
    for i in range(cnt):
        podviborka = []
        for j in range(sample.size):
            val = np.random.uniform(teta, 2 * teta)
            podviborka.append(val)
        podviborka = np.array(podviborka)
        answ.append(param_func(podviborka))
    return np.array(answ)

In [22]:
bootstrap_param_tetas = do_parametric_bootstrap(50000, sample, get_teta, model_teta)
deltas = bootstrap_param_tetas - model_teta

variation_series_delta = sorted(deltas)

k1 = int((1 - beta) * 0.5 * 50000)
k2 = int((1 + beta) * 0.5 * 50000)

left_bound_param_boots = teta_1 - variation_series_delta[k2]
right_bound_param_boots = teta_1 - variation_series_delta[k1]
print(f"бустраповский (параметрический) доверительный интервал по ОММ: {left_bound_param_boots} < teta < {right_bound_param_boots}")
print(f"его длина: {right_bound_param_boots - left_bound_param_boots}")

бустраповский (параметрический) доверительный интервал по ОММ: 9.538604718397963 < teta < 10.2887383078086
его длина: 0.7501335894106376


In [10]:
def calc_teta_3(sample):
    return np.max(sample) / 2 * (2 * n + 2) / (2 * n + 1)

In [29]:
bootstrap_tetas = do_bootstrap(1000, sample, calc_teta_3)
# bootstrap_tetas1 = do_bootstrap(1000, sample, calc_teta_3)
# bs1 = sorted(bootstrap_tetas)
# bs2 = sorted(bootstrap_tetas1)
# for i in range(n):
#     if bs1[i] != bs2[i]:
#         print(i)
model_teta_2 = calc_teta_3(sample)

deltas = bootstrap_tetas - model_teta_2

k1 = int((1 - beta) * 0.5 * 1000)
k2 = int((1 + beta) * 0.5 * 1000)

variation_series = sorted(deltas)

left_bound_unparam_boots = model_teta_2 - variation_series[k2]
right_bound_unparam_boots = model_teta_2 - variation_series[k1]
print(variation_series[k1], variation_series[k2], k1, k2)

print(f"бустраповский (непараметрический) доверительный интервал по ОМП: {left_bound_unparam_boots} < teta < {right_bound_unparam_boots}")
print(f"его длина: {right_bound_unparam_boots - left_bound_unparam_boots}")

-0.15058046448229945 0.0 25 975
бустраповский (непараметрический) доверительный интервал по ОМП: 10.006556873113563 < teta < 10.157137337595863
его длина: 0.15058046448229945


In [31]:
bootstrap_param_tetas = do_parametric_bootstrap(50000, sample, calc_teta_3, model_teta_2)
deltas = bootstrap_param_tetas - model_teta_2

variation_series_delta = sorted(deltas)

k1 = int((1 - beta) * 0.5 * 50000)
k2 = int((1 + beta) * 0.5 * 50000)

left_bound_param_boots = teta_1 - variation_series_delta[k2]
right_bound_param_boots = teta_1 - variation_series_delta[k1]
print(f"бустраповский (параметрический) доверительный интервал по ОМП: {left_bound_param_boots} < teta < {right_bound_param_boots}")
print(f"его длина: {right_bound_param_boots - left_bound_param_boots}")

бустраповский (параметрический) доверительный интервал по ОМП: 9.866240562541766 < teta < 10.050203018958289
его длина: 0.18396245641652342
